# Garmin e CPAP

Correlação entre o relógio Garmin e o CPAP. O Garmin não substitui as métricas respiratórias: ele acrescenta frequência cardíaca, HRV, estresse, fases, score e SpO2 de pulso. Também cobre noites sem o aparelho.

A `calendarDate` do Garmin é o dia em que acordei. A noite da ResMed é o dia anterior (`night = calendarDate − 1`). Séries em UTC, comparadas em `America/Sao_Paulo`.

SpO2 de pulso é tendência, não oximetria clínica. O detalhe por minuto só existe a partir de maio/2026. Sem teste de significância.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

from sleep_analysis.db import connect
from sleep_analysis.paths import PROJECT_DIR

plt.rcParams.update({"figure.figsize": (11, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
REPORT_DIR = PROJECT_DIR / "data/reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_TZ = "America/Sao_Paulo"
CPAP_START = pd.Timestamp("2025-09-16")
MIN_PAIRS = 30

db = connect()

def local_from_utc(timestamps):
    return pd.to_datetime(timestamps, utc=True).dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)

garmin = db.sql("SELECT * FROM garmin_nights ORDER BY night").df()
nights = db.sql("""
    SELECT night, usage_hours, pressure, ahi, leak_95_l_min, resp_rate_50, first_mask_on
    FROM nights
""").df()
flow = db.sql("""
    SELECT night, surge_index, hypopnea30_index, flow_limited_pct,
           wake_like_bouts, resp_rate_median, stable_pct
    FROM night_metrics
""").df()
for frame in (garmin, nights, flow):
    frame["night"] = pd.to_datetime(frame["night"])
garmin["sleep_start_local"] = local_from_utc(garmin.sleep_start_utc)
print(f"Garmin: {len(garmin)} noites ({garmin.night.min():%d/%m/%Y} a {garmin.night.max():%d/%m/%Y}); "
      f"detalhe por minuto em {int(garmin.has_detail.sum())}")


## Pareamento

O início do sono no Garmin deve cair pouco depois de `first_mask_on`. A respiração média do relógio deve acompanhar a frequência do CPAP: é a checagem de que as noites são as mesmas.


In [ ]:
paired = garmin.merge(nights, on="night", how="inner")
used = paired[paired.usage_hours >= 4].copy()
used["start_delta_min"] = (used.sleep_start_local - used.first_mask_on).dt.total_seconds() / 60
delta = used.start_delta_min.dropna()
print(f"Noites em comum com uso ≥ 4 h: {len(used)}; com os dois horários: {len(delta)}")
print(f"Garmin menos máscara (min): mediana {delta.median():.0f}, "
      f"Q1–Q3 {delta.quantile(.25):.0f}–{delta.quantile(.75):.0f}")
breathing = used.dropna(subset=["respiration_avg", "resp_rate_50"])
breath_rho = stats.spearmanr(breathing.respiration_avg, breathing.resp_rate_50).statistic
print(f"Respiração Garmin × frequência do CPAP: ρ = {breath_rho:.2f} (n = {len(breathing)})")


## Spearman, pressão de 7,0 cmH₂O

Só noites com pelo menos 4 h e pressão de 7,0 cmH₂O, para não misturar ajustes. Pares com menos de 30 noites ficam em branco. HRV e o detalhe do fluxo não cobrem o período inteiro, então o `n` muda por célula.


In [ ]:
GARMIN_COLUMNS = {
    "sleep_hours": "Sono (h)", "sleep_score": "Score", "resting_heart_rate": "FC de repouso",
    "avg_overnight_hrv": "HRV", "spo2_avg": "SpO2 média", "spo2_lowest": "SpO2 mínima",
    "respiration_avg": "Respiração", "sleep_stress": "Estresse", "body_battery_change": "Body Battery",
    "rem_hours": "REM (h)", "deep_hours": "Profundo (h)",
}
CPAP_COLUMNS = {
    "ahi": "AHI", "leak_95_l_min": "Vazamento p95", "resp_rate_50": "FR CPAP",
    "usage_hours": "Uso (h)", "surge_index": "Surge", "hypopnea30_index": "Hypopnea30",
    "flow_limited_pct": "Fluxo limitado", "wake_like_bouts": "Blocos de vigília",
    "resp_rate_median": "FR do fluxo",
}

cohort = used.merge(flow, on="night", how="left")
cohort = cohort[np.isclose(cohort.pressure, 7.0)]
rows = []
for garmin_column, garmin_label in GARMIN_COLUMNS.items():
    for cpap_column, cpap_label in CPAP_COLUMNS.items():
        pair = cohort[[garmin_column, cpap_column]].dropna()
        rho = stats.spearmanr(pair[garmin_column], pair[cpap_column]).statistic if len(pair) >= MIN_PAIRS else np.nan
        rows.append({"garmin": garmin_label, "cpap": cpap_label, "n": len(pair), "rho": rho})
correlations = pd.DataFrame(rows)
rho_table = correlations.pivot(index="garmin", columns="cpap", values="rho").reindex(
    index=GARMIN_COLUMNS.values(), columns=CPAP_COLUMNS.values())
count_table = correlations.pivot(index="garmin", columns="cpap", values="n").reindex(
    index=GARMIN_COLUMNS.values(), columns=CPAP_COLUMNS.values())
display(Markdown(f"Coorte: **{len(cohort)} noites** a 7,0 cmH₂O com uso ≥ 4 h."))
display(rho_table.round(2))
rho_table.round(3).to_csv(REPORT_DIR / "06_spearman_rho.csv")
count_table.to_csv(REPORT_DIR / "06_spearman_n.csv")

fig, ax = plt.subplots(figsize=(11, 6.2))
image = ax.imshow(rho_table.to_numpy(), cmap="RdBu_r", vmin=-0.7, vmax=0.7, aspect="auto")
ax.set_xticks(range(rho_table.shape[1]), rho_table.columns, rotation=40, ha="right")
ax.set_yticks(range(rho_table.shape[0]), rho_table.index)
for row_index, garmin_label in enumerate(rho_table.index):
    for column_index, cpap_label in enumerate(rho_table.columns):
        value = rho_table.loc[garmin_label, cpap_label]
        count = int(count_table.loc[garmin_label, cpap_label])
        ax.text(column_index, row_index, "·" if pd.isna(value) else f"{value:.2f}\nn={count}",
                ha="center", va="center", fontsize=7.5, color="#222222")
ax.set_title("Spearman: Garmin × CPAP, 7,0 cmH₂O e uso ≥ 4 h")
fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02, label="ρ")
fig.tight_layout()
fig.savefig(REPORT_DIR / "06_spearman.png", dpi=160, bbox_inches="tight")
plt.show()


## Com CPAP e sem CPAP

Três grupos no Garmin: antes de 16/09/2025, uso de pelo menos 4 h e noites da era do CPAP sem uso registrado. Uso parcial fica de fora da tabela. Mediana e Q1–Q3; os grupos não são pareados por época.


In [ ]:
labeled = garmin.merge(nights[["night", "usage_hours"]], on="night", how="left")

def usage_group(row):
    if row.night < CPAP_START:
        return "antes do CPAP"
    if pd.notna(row.usage_hours) and row.usage_hours >= 4:
        return "CPAP ≥ 4 h"
    if pd.notna(row.usage_hours) and row.usage_hours > 0:
        return "uso parcial"
    return "sem uso na era CPAP"

labeled["grupo"] = labeled.apply(usage_group, axis=1)
GROUP_COLUMNS = ["sleep_hours", "sleep_score", "resting_heart_rate", "avg_overnight_hrv",
                 "spo2_avg", "spo2_lowest", "respiration_avg", "sleep_stress", "rem_hours"]
GROUP_ORDER = ["antes do CPAP", "sem uso na era CPAP", "CPAP ≥ 4 h"]
parts = []
for column in GROUP_COLUMNS:
    grouped = labeled.groupby("grupo")[column]
    part = pd.DataFrame({
        "mediana": grouped.median(),
        "Q1": grouped.quantile(0.25),
        "Q3": grouped.quantile(0.75),
        "n": grouped.count(),
    })
    part["métrica"] = column
    parts.append(part.reset_index())
groups = pd.concat(parts, ignore_index=True)
groups = groups[groups.grupo.isin(GROUP_ORDER)]
wide_groups = groups.pivot(index="métrica", columns="grupo", values="mediana").reindex(index=GROUP_COLUMNS, columns=GROUP_ORDER)
count_groups = groups.pivot(index="métrica", columns="grupo", values="n").reindex(index=GROUP_COLUMNS, columns=GROUP_ORDER)
display(wide_groups.round(2))
display(Markdown("n por célula"))
display(count_groups)
groups.to_csv(REPORT_DIR / "06_cpap_groups.csv", index=False)
print(labeled.grupo.value_counts().reindex(GROUP_ORDER + ["uso parcial"]).to_string())


## Dentro da noite

A frequência cardíaca do Garmin durante o sono vem a cada 2 minutos; a SpO2, a cada minuto. Por isso a janela é de 10 minutos antes e 10 depois de cada evento: com 2 minutos, quase sempre há uma única leitura de cada lado e a mediana não sai do lugar.

A diferença é a mediana depois menos a mediana antes. Também conto a fração de eventos em que essa diferença passa de 5 bpm ou de 2 pontos de SpO2. Eventos do aparelho e eventos inferidos do fluxo entram separados. A diferença é descritiva: o evento e a mudança do sinal podem ter a mesma causa.

Para a FC, `pct_movimento` é a fração com mediana pelo menos 5 bpm mais alta. Para a SpO2, é a fração com mediana pelo menos 2 pontos mais baixa.


In [ ]:
def change_around(events, samples, value_column, window_s=600):
    rows = []
    series_by_night = {night: group.sort_values("timestamp_local") for night, group in samples.groupby("night")}
    window = np.timedelta64(window_s, "s")
    for event in events.itertuples(index=False):
        series = series_by_night.get(event.night)
        if series is None:
            continue
        times = series.timestamp_local.to_numpy()
        values = series[value_column].to_numpy()
        moment = np.datetime64(event.timestamp)
        before = values[(times >= moment - window) & (times < moment)]
        after = values[(times >= moment) & (times < moment + window)]
        if len(before) == 0 or len(after) == 0:
            continue
        rows.append({"night": event.night, "event_type": event.event_type,
                     "delta": float(np.median(after) - np.median(before))})
    return pd.DataFrame(rows)

def load_series(table, value_column):
    frame = db.sql(f"SELECT night, timestamp_utc, {value_column} FROM {table}").df()
    frame["night"] = pd.to_datetime(frame.night)
    frame["timestamp_local"] = local_from_utc(frame.timestamp_utc)
    return frame

heart_rate = load_series("garmin_hr", "heart_rate_bpm")
spo2 = load_series("garmin_spo2", "spo2_percent")
device_events = db.sql("SELECT night, timestamp, event_type FROM events").df()
inferred_events = db.sql("""
    SELECT night, timestamp, event_type FROM respiratory_events
    WHERE event_type IN ('surge', 'hypopnea30')
""").df()
device_events["night"] = pd.to_datetime(device_events.night)
inferred_events["night"] = pd.to_datetime(inferred_events.night)

def summarize_shift(events, samples, value_column, source, unit_jump, drop=False):
    shift = change_around(events, samples, value_column)
    columns = ["fonte", "evento", "n_eventos", "n_noites", "mediana", "Q1", "Q3", "pct_movimento"]
    if shift.empty:
        return pd.DataFrame(columns=columns)
    rows = []
    for event_type, group in shift.groupby("event_type"):
        rows.append({
            "fonte": source, "evento": event_type, "n_eventos": len(group),
            "n_noites": group.night.nunique(), "mediana": group.delta.median(),
            "Q1": group.delta.quantile(0.25), "Q3": group.delta.quantile(0.75),
            "pct_movimento": 100 * ((group.delta <= -unit_jump) if drop else (group.delta >= unit_jump)).mean(),
        })
    return pd.DataFrame(rows, columns=columns)

heart_shift = pd.concat([
    summarize_shift(device_events, heart_rate, "heart_rate_bpm", "FC aparelho", 5),
    summarize_shift(inferred_events, heart_rate, "heart_rate_bpm", "FC fluxo", 5),
    summarize_shift(device_events, spo2, "spo2_percent", "SpO2 aparelho", 2, drop=True),
    summarize_shift(inferred_events, spo2, "spo2_percent", "SpO2 fluxo", 2, drop=True),
], ignore_index=True)
display(heart_shift.round(2))
heart_shift.round(3).to_csv(REPORT_DIR / "06_signal_around_events.csv", index=False)


## Véspera

`garmin_days.day` é o dia civil. Ele coincide com a noite da ResMed: os passos e o estresse de 16/10 descrevem o dia anterior ao sono que o CPAP chama de noite de 16/10.

Exercício tardio é uma atividade, fora `breathwork`, que termina nas 3 h antes do início do sono no Garmin.


In [ ]:
days = db.sql("SELECT * FROM garmin_days").df()
activities = db.sql("SELECT * FROM garmin_activities").df()
days["day"] = pd.to_datetime(days.day)
activities["end_local"] = activities.start_local + pd.to_timedelta(activities.duration_s, unit="s")
day_and_night = garmin.merge(days, left_on="night", right_on="day", how="inner", suffixes=("", "_day"))

def late_exercise(sleep_start):
    if pd.isna(sleep_start):
        return False
    window_start = sleep_start - pd.Timedelta(hours=3)
    finished = activities[(activities.end_local > window_start) & (activities.end_local <= sleep_start)]
    finished = finished[finished.activity_type.fillna("") != "breathwork"]
    return not finished.empty

day_and_night["late_exercise"] = day_and_night.sleep_start_local.map(late_exercise)
day_and_night["intensity_minutes"] = (
    day_and_night.moderate_intensity_minutes.fillna(0) + day_and_night.vigorous_intensity_minutes.fillna(0)
)

DAY_COLUMNS = {"steps": "Passos", "intensity_minutes": "Minutos de intensidade", "stress_avg": "Estresse do dia"}
NIGHT_COLUMNS = {"sleep_score": "Score", "sleep_hours": "Sono (h)", "resting_heart_rate": "FC de repouso", "avg_overnight_hrv": "HRV"}
day_rows = []
for day_column, day_label in DAY_COLUMNS.items():
    for night_column, night_label in NIGHT_COLUMNS.items():
        pair = day_and_night[[day_column, night_column]].dropna()
        rho = stats.spearmanr(pair[day_column], pair[night_column]).statistic if len(pair) >= MIN_PAIRS else np.nan
        day_rows.append({"véspera": day_label, "noite": night_label, "n": len(pair), "rho": rho})
day_correlation = pd.DataFrame(day_rows)
display(day_correlation.pivot(index="véspera", columns="noite", values="rho").reindex(
    index=DAY_COLUMNS.values(), columns=NIGHT_COLUMNS.values()).round(2))
day_correlation.round(3).to_csv(REPORT_DIR / "06_previous_day.csv", index=False)

with_exercise = day_and_night[day_and_night.late_exercise]
without_exercise = day_and_night[~day_and_night.late_exercise]
exercise_rows = []
for column, label in (("sleep_score", "Score"), ("sleep_hours", "Sono (h)"), ("resting_heart_rate", "FC de repouso")):
    exercise_rows.append({
        "métrica": label,
        "com exercício tardio": with_exercise[column].median(),
        "n com": with_exercise[column].count(),
        "sem": without_exercise[column].median(),
        "n sem": without_exercise[column].count(),
    })
exercise_summary = pd.DataFrame(exercise_rows)
display(exercise_summary.round(2))
exercise_summary.to_csv(REPORT_DIR / "06_late_exercise.csv", index=False)
print(f"Noites com atividade (exceto breathwork) terminando nas 3 h anteriores: {int(day_and_night.late_exercise.sum())} de {len(day_and_night)}")
db.close()


## Como ler

- Respiração do Garmin alinhada à do CPAP confirma o pareamento. Correlação perto de zero com AHI ou SpO2 mínima significa que o relógio não mede o resíduo respiratório.
- FC de repouso, HRV, estresse e Body Battery descrevem a noite junto com o fluxo; não explicam a causa.
- A comparação com e sem CPAP mistura épocas e hábitos. Serve como contexto do sono sem aparelho, não como efeito do tratamento.
- Nos 10 minutos ao redor do evento, a mediana da FC não sobe e a da SpO2 não cai. Uma alta de 5 bpm, ou uma queda de 2 pontos de SpO2, aparece em poucos eventos. Nesse intervalo, o relógio não marca o evento respiratório.
- Exercício tardio e estresse do dia são o começo do objetivo de médio prazo. A amostra de exercício perto de deitar pode ser pequena.
